# Day 57 — TFRecord, Proto Buffers
- Explain why TFRecord is useful for storing large datasets.
- Write and read serialized examples.
- Use GZIP compression.
- Understand Protocol Buffers and TensorFlow's `tf.train` message types.
- Parse `tf.train.Example` features.
- Recognize when `SequenceExample` is appropriate.

## 1. Why TFRecord?

A **TFRecord** file is a sequence of records. Each record is typically a byte string, often a serialized Protocol Buffer message.

Think of it as a container format:
- **TFRecord** handles record boundaries and sequential reading.
- **Protocol Buffers (protobufs)** define how structured data is serialized into bytes.
- **`tf.data`** provides a pipeline for reading and transforming the records.

TFRecord is useful when datasets are large, contain varied data, or benefit from sequential/sharded storage. It does not automatically make data smaller or faster in every situation.

## 2. Imports and version

In [10]:
import tensorflow as tf
import numpy as np
from pathlib import Path

print("TensorFlow:", tf.__version__)
WORK_DIR = Path("day57_tfrecord_data")
WORK_DIR.mkdir(exist_ok=True)


TensorFlow: 2.21.0


## 3. A first TFRecord file

`tf.io.TFRecordWriter` writes byte strings as records. Here we write a few plain-text records to see the container mechanics. In real pipelines, records commonly contain serialized protobufs.

In [11]:
plain_path = WORK_DIR / "plain_records.tfrecord"

with tf.io.TFRecordWriter(str(plain_path)) as writer:
    for message in [b"first record", b"second record", b"third record"]:
        writer.write(message)

raw_dataset = tf.data.TFRecordDataset([str(plain_path)])
for record in raw_dataset:
    print(record.numpy())


b'first record'
b'second record'
b'third record'


### Key idea
The reader returns tensors of type `tf.string`; each tensor contains the bytes for one record. TFRecord itself does not tell TensorFlow what those bytes mean—you choose a schema and parse them accordingly.

## 4. Compressed TFRecords

Compression can reduce storage and I/O, at the cost of CPU work for compression/decompression. Writer and reader must use matching compression settings.

In [12]:
compressed_path = WORK_DIR / "compressed_records.tfrecord"

options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter(str(compressed_path), options=options) as writer:
    for i in range(5):
        writer.write(f"sample-{i}".encode("utf-8"))

compressed_dataset = tf.data.TFRecordDataset(
    [str(compressed_path)],
    compression_type="GZIP"
)
print([x.numpy().decode("utf-8") for x in compressed_dataset])


['sample-0', 'sample-1', 'sample-2', 'sample-3', 'sample-4']


## 5. Protocol Buffers and `tf.train.Example`

Protocol Buffers are a language-neutral way to serialize structured messages. TensorFlow provides protobuf message classes under `tf.train`.

An `Example` contains a map from feature names to values. Each feature uses one of three common value-list types:
- `BytesList`: byte strings (often text or encoded images)
- `FloatList`: floating-point numbers
- `Int64List`: integer values

The schema is defined by your feature names and expected types; parsing needs to match it.

In [13]:
example = tf.train.Example(
    features=tf.train.Features(
        feature={
            "age": tf.train.Feature(
                int64_list=tf.train.Int64List(value=[35])
            ),
            "height_m": tf.train.Feature(
                float_list=tf.train.FloatList(value=[1.78])
            ),
            "city": tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[b"Bengaluru"])
            ),
        }
    )
)

serialized = example.SerializeToString()
print("Serialized bytes:", len(serialized))
print(serialized[:30], "...")


Serialized bytes: 61
b'\n;\n\x0c\n\x03age\x12\x05\x1a\x03\n\x01#\n\x15\n\x04city\x12\r\n\x0b\n\t' ...


## 6. Parse a serialized Example

`tf.io.parse_single_example` converts one serialized `Example` into tensors according to a feature specification. Use `FixedLenFeature` for fixed-size values and specify the correct dtype.

In [14]:
feature_spec = {
    "age": tf.io.FixedLenFeature([], tf.int64),
    "height_m": tf.io.FixedLenFeature([], tf.float32),
    "city": tf.io.FixedLenFeature([], tf.string),
}

parsed = tf.io.parse_single_example(serialized, feature_spec)
{key: value.numpy() for key, value in parsed.items()}


{'age': np.int64(35), 'city': b'Bengaluru', 'height_m': np.float32(1.78)}

### Write and read a dataset of Examples

Now combine protobuf serialization with TFRecord storage, then parse records in a `tf.data` pipeline.

In [15]:
examples_path = WORK_DIR / "people.tfrecord"

def make_example(age, height, city):
    return tf.train.Example(features=tf.train.Features(feature={
        "age": tf.train.Feature(int64_list=tf.train.Int64List(value=[age])),
        "height_m": tf.train.Feature(float_list=tf.train.FloatList(value=[height])),
        "city": tf.train.Feature(bytes_list=tf.train.BytesList(value=[city.encode()])),
    }))

with tf.io.TFRecordWriter(str(examples_path)) as writer:
    for age, height, city in [
        (24, 1.68, "Mysuru"),
        (35, 1.78, "Bengaluru"),
        (42, 1.73, "Chennai"),
    ]:
        writer.write(make_example(age, height, city).SerializeToString())

def parse_person(serialized_example):
    return tf.io.parse_single_example(serialized_example, feature_spec)

people_ds = tf.data.TFRecordDataset([str(examples_path)]).map(parse_person)
for person in people_ds:
    print({k: v.numpy() for k, v in person.items()})


{'age': np.int64(24), 'city': b'Mysuru', 'height_m': np.float32(1.68)}
{'age': np.int64(35), 'city': b'Bengaluru', 'height_m': np.float32(1.78)}
{'age': np.int64(42), 'city': b'Chennai', 'height_m': np.float32(1.73)}


## 7. Missing, variable-length, and fixed-length features

A `FixedLenFeature` expects a known shape. A default value can be supplied for a feature that may be absent. For variable-length lists, use `VarLenFeature`, which produces a sparse tensor.

In [16]:
optional_spec = {
    "age": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
    "tags": tf.io.VarLenFeature(tf.string),
}

example_with_tags = tf.train.Example(features=tf.train.Features(feature={
    "age": tf.train.Feature(int64_list=tf.train.Int64List(value=[29])),
    "tags": tf.train.Feature(bytes_list=),tf.train.BytesList(
        value=[b"python", b"tensorflow", b"ml"]
    )
}))
parsed_optional = tf.io.parse_single_example(
    example_with_tags.SerializeToString(), optional_spec
)
print("age:", parsed_optional["age"].numpy())
print("tags:", tf.sparse.to_dense(parsed_optional["tags"]).numpy())


age: 29
tags: [b'python' b'tensorflow' b'ml']


## 8. `SequenceExample`: data with time steps

Use `tf.train.SequenceExample` when each sample contains:
- **Context features** that describe the whole example (e.g., user ID or label).
- **Feature lists** that vary over time (e.g., sensor readings per second, words in a sentence, frames in a clip).

A regular `Example` is suitable for one record of features. `SequenceExample` adds a structured sequence of feature values.

In [17]:
sequence = tf.train.SequenceExample(
    context=tf.train.Features(feature={
        "device": tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[b"sensor-A"])
        ),
        "session_id": tf.train.Feature(
            int64_list=tf.train.Int64List(value=[101])
        ),
    }),
    feature_lists=tf.train.FeatureLists(feature_list={
        "temperature": tf.train.FeatureList(feature=[
            tf.train.Feature(float_list=tf.train.FloatList(value=[21.1])),
            tf.train.Feature(float_list=tf.train.FloatList(value=[21.4])),
            tf.train.Feature(float_list=tf.train.FloatList(value=[21.8])),
        ]),
        "status": tf.train.FeatureList(feature=[
            tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"ok"])),
            tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"ok"])),
            tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"warning"])),
        ]),
    })
)
sequence_bytes = sequence.SerializeToString()
print("Serialized SequenceExample bytes:", len(sequence_bytes))


Serialized SequenceExample bytes: 137


### Parse the context and sequence

`tf.io.parse_single_sequence_example` accepts separate specifications for context and sequence data. Sequence features are returned as sparse tensors by default, which can be converted to dense tensors when appropriate.

In [18]:
context_spec = {
    "device": tf.io.FixedLenFeature([], tf.string),
    "session_id": tf.io.FixedLenFeature([], tf.int64),
}
sequence_spec = {
    "temperature": tf.io.FixedLenSequenceFeature([], tf.float32),
    "status": tf.io.FixedLenSequenceFeature([], tf.string),
}

context_out, sequence_out = tf.io.parse_single_sequence_example(
    sequence_bytes,
    context_features=context_spec,
    sequence_features=sequence_spec,
)
print("context:", {k: v.numpy() for k, v in context_out.items()})
print("temperatures:", sequence_out["temperature"].numpy())
print("statuses:", sequence_out["status"].numpy())


context: {'device': b'sensor-A', 'session_id': np.int64(101)}
temperatures: [21.1 21.4 21.8]
statuses: [b'ok' b'ok' b'warning']


## Quick reference sheet

| Concept | Remember |
|---|---|
| TFRecord | Stores a sequence of byte records; does not define their meaning |
| `TFRecordWriter` | Writes each serialized record |
| `TFRecordDataset` | Reads records into a `tf.data` pipeline |
| Compression | Writer and reader settings must match |
| Protocol Buffer | Structured message serialization format |
| `tf.train.Feature` | Holds bytes, float, or integer values |
| `tf.train.Example` | Map of named features for one example |
| `parse_single_example` | Parses one serialized Example using a feature spec |
| `SequenceExample` | Context plus per-time-step feature lists |
| `parse_single_sequence_example` | Parses context and sequential features |

